# 04 — Graphiques Plotly : accidents d'aviation (NTSB)
Notebook de visualisation à partir de `data/aviation.db` (États-Unis, 2008-2026).

**À savoir :** 2026 est une année partielle, elle est signalée sur les graphiques d'évolution.

In [10]:
import sqlite3
from pathlib import Path

import pandas as pd
import plotly.express as px

# Le notebook est dans notebooks/, la base dans data/
DB = next(p for p in [Path("../data/aviation.db"), Path("data/aviation.db")] if p.exists())
FIG = DB.parent.parent / "figures"
FIG.mkdir(exist_ok=True)
con = sqlite3.connect(DB)

def sql(q):
    return pd.read_sql_query(q, con)

def sauver(fig, nom):
    fig.write_html(FIG / f"{nom}.html")
    fig.show()

print("Base :", DB.resolve())

Base : /Users/nada/projet-aviation/data/aviation.db


## 0. Contrôle du schéma
Si une requête plus bas échoue, la cause est presque toujours un nom de colonne : cette cellule les liste.

In [11]:
for (t,) in con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"):
    cols = [r[1] for r in con.execute(f"PRAGMA table_info({t})")]
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t} ({n} lignes) : {', '.join(cols)}\n")

aircraft (25923 lignes) : ev_id, aircraft_key, categorie, far_part, code_segment, id_constructeur, modele, annee_construction, nb_moteurs, dommages, type_vol

constructeurs (3685 lignes) : id_constructeur, nom

etats (58 lignes) : code_etat, nom, region_faa

events (25510 lignes) : ev_id, ntsb_no, ev_type, ev_date, ev_year, ev_month, ev_city, code_etat, ev_country, latitude, longitude, gravite_max, morts, blesses_graves, blesses_legers, indemnes, total_personnes, lumiere, meteo

findings (73089 lignes) : id_finding, ev_id, aircraft_key, finding_no, finding_code, description, cause_factor

injuries (157156 lignes) : id_injury, ev_id, aircraft_key, categorie_personne, niveau, nombre

phases (48 lignes) : phase_no, libelle, famille

segments (8 lignes) : code_segment, libelle

sequences (61101 lignes) : id_sequence, ev_id, aircraft_key, occurrence_no, phase_no, evenement, description, evenement_definissant

sqlite_sequence (3 lignes) : name, seq



## 1. Évolution du nombre d'événements par année

In [12]:
df = sql("""SELECT ev_year AS annee, COUNT(*) AS evenements,
                    SUM(gravite_max='FATL') AS mortels
             FROM events GROUP BY ev_year ORDER BY ev_year""")
df["non_mortels"] = df["evenements"] - df["mortels"]
df["note"] = df["annee"].map(lambda a: "année partielle" if a == df["annee"].max() else "")

fig = px.bar(df, x="annee", y=["non_mortels", "mortels"],
             labels={"value": "Événements", "annee": "Année", "variable": ""},
             title="Événements par année (la dernière année est partielle)",
             color_discrete_map={"non_mortels": "#9bb7d4", "mortels": "#c0392b"})
fig.update_layout(barmode="stack", legend_title_text="")
sauver(fig, "01_evolution_annuelle")

## 2. Gravité des événements

In [13]:
df = sql("""SELECT COALESCE(gravite_max,'Non renseigné') AS gravite, COUNT(*) AS evenements
             FROM events GROUP BY 1 ORDER BY evenements DESC""")
noms = {"NONE": "Aucun blessé", "MINR": "Blessés légers", "SERS": "Blessés graves", "FATL": "Mortel"}
df["gravite"] = df["gravite"].replace(noms)
fig = px.bar(df, x="gravite", y="evenements", text="evenements",
             labels={"gravite": "", "evenements": "Événements"},
             title="Gravité maximale des événements")
sauver(fig, "02_gravite")

## 3. Segments d'exploitation : volume et taux de mortalité
L'aviation générale concentre le volume ; les compagnies régulières (Part 121) ont très peu d'accidents mortels.

In [14]:
df = sql("""SELECT s.libelle AS segment, COUNT(*) AS evenements,
                    SUM(e.gravite_max='FATL') AS mortels
             FROM events e
             JOIN (SELECT ev_id, MIN(code_segment) AS code_segment FROM aircraft GROUP BY ev_id) a USING(ev_id)
             JOIN segments s USING(code_segment)
             GROUP BY s.libelle ORDER BY evenements DESC""")
df["taux_mortel_%"] = (100 * df["mortels"] / df["evenements"]).round(1)
display(df)

fig = px.bar(df, x="evenements", y="segment", orientation="h", color="taux_mortel_%",
             color_continuous_scale="Reds", text="evenements",
             labels={"evenements": "Événements", "segment": "", "taux_mortel_%": "% mortels"},
             title="Événements par segment (couleur = part d'accidents mortels)")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
sauver(fig, "03_segments")

,segment,evenements,mortels,taux_mortel_%
0,"Aviation générale (Part 91, dont 91K)",22197,3843,17.3
1,Travail aérien (Parts 133 et 137),1326,177,13.3
2,Taxi aérien et commuter (Part 135),870,156,17.9
3,Compagnies régulières (Part 121),681,9,1.3
4,Aéronefs publics et militaires,297,55,18.5
5,"Exploitants étrangers (Part 129, non-U.S.)",88,5,5.7
6,Non renseigné,37,3,8.1
7,"Régimes spéciaux (drones, ultralégers, expérim...",14,4,28.6


## 4. Constructeurs les plus représentés

In [15]:
df = sql("""SELECT c.nom AS constructeur, COUNT(*) AS aeronefs
             FROM aircraft a JOIN constructeurs c USING(id_constructeur)
             GROUP BY c.nom ORDER BY aeronefs DESC LIMIT 10""")
fig = px.bar(df, x="aeronefs", y="constructeur", orientation="h", text="aeronefs",
             labels={"aeronefs": "Aéronefs impliqués", "constructeur": ""},
             title="Top 10 des constructeurs (nombre d'aéronefs impliqués, pas un taux de risque)")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
sauver(fig, "04_constructeurs")

## 5. Phases de vol
Cette cellule dépend des noms de colonnes de `phases`. Si elle affiche un message, envoie-moi la sortie de la cellule 0.

In [16]:
cols = [r[1] for r in con.execute("PRAGMA table_info(phases)")]
col_famille = next((c for c in cols if "famille" in c.lower()), None)
if col_famille is None:
    print("Pas de colonne 'famille' dans phases. Colonnes :", cols)
else:
    df = sql(f"""SELECT p.{col_famille} AS famille, COUNT(*) AS lignes
                  FROM sequences s JOIN phases p USING(phase_no)
                  GROUP BY 1 ORDER BY lignes DESC""")
    fig = px.bar(df, x="lignes", y="famille", orientation="h", text="lignes",
                 labels={"lignes": "Séquences d'événements", "famille": ""},
                 title="Séquences d'accident par phase de vol")
    fig.update_layout(yaxis={"categoryorder": "total ascending"})
    sauver(fig, "05_phases")

## 6. Carte des accidents mortels
4 000 points environ : lisible et rapide à afficher.

In [17]:
cols = [r[1] for r in con.execute("PRAGMA table_info(events)")]
col_etat = next((c for c in ["code_etat", "ev_state", "etat"] if c in cols), None)
print("Colonne État utilisée :", col_etat)

select_etat = f", e.{col_etat} AS etat" if col_etat else ""
df = sql(f"""SELECT e.ev_id, e.ev_year{select_etat}, e.latitude, e.longitude
             FROM events e
             WHERE e.gravite_max='FATL' AND e.latitude IS NOT NULL AND e.longitude IS NOT NULL""")

hover = ["ev_year"] + (["etat"] if col_etat else [])
fig = px.scatter_map(df, lat="latitude", lon="longitude", hover_data=hover,
                     zoom=3, height=600, opacity=0.5, map_style="open-street-map",
                     title=f"Accidents mortels géolocalisés ({len(df)})")
sauver(fig, "06_carte_mortels")

Colonne État utilisée : code_etat


## Limites à citer dans le rapport
- Périmètre : événements aux États-Unis uniquement (25 510 sur 31 453).
- 2026 est une année incomplète.
- Le nombre d'aéronefs par constructeur reflète la taille de la flotte, pas la dangerosité.
- Les événements sans coordonnées ne figurent pas sur la carte.